# HOD with the Background Subtraction Technique — quick start

This notebook measures the Halo Occupation Distribution $\langle N|M\rangle$ of a galaxy
catalogue with `bst_hod`. Set `CATALOG` to your table (e.g. the cosmoDC2 table
`mock_21_large_FZ_z05_cosmoDC2.mpeg`). If the file is not found, a small synthetic
catalogue is used so you can run everything.

In [ ]:
import os
import matplotlib.pyplot as plt
from bst_hod import compute_hod, compute_hod_samples, hod_curve, find_central_galaxies, plot_hod, load_catalog

## 1. The galaxy table

Required columns: galaxy id, RA, Dec (degrees), redshift and apparent r magnitude.
For mocks also `halo_id`, `loghalo_mass` and `is_central`. The default names are the ones of
the original DC2 tables (`galid, ra, dec, z, mr_lsst, halo_id, loghalo_mass, is_central`);
use `columns={...}` to map other names, e.g. photometric redshifts:
`columns={"z": "photoz_mode", "mag": "mag_r_photoz"}`.

In [ ]:
CATALOG = "Tabs/mock_21_large_FZ_z05_cosmoDC2.mpeg"   # <- your table

if os.path.exists(CATALOG):
    cat = load_catalog(CATALOG)
else:
    from bst_hod.synthetic import make_mock_catalog
    print(f"{CATALOG} not found: using a synthetic catalogue")
    cat = load_catalog(make_mock_catalog())
cat.head()

## 2. One volume-limited sample

`compute_hod` returns one row per group centre with the background-subtracted count
`N_bst` and, for mocks, the true count `N_true`. The defaults reproduce the original
DC2 run (aperture = R200 clipped to [0.05, 1.25] Mpc, ring 1.5–3.5 × aperture,
centres with M < −19.5, the original z cuts).

In [ ]:
res = compute_hod(cat, mlim=-19)
res.head()

In [ ]:
hod = hod_curve(res, errors="bootstrap")     # or errors="jackknife"
hod

## 3. Several magnitude limits

In [ ]:
samples = compute_hod_samples(cat, mlims=[-18, -19, -20], verbose=False)
curves = {f"M < {m}": hod_curve(r) for m, r in samples.items()}
ax = plot_hod(curves)
plt.show()

## 4. Without membership information: Central Galaxy Finder

For a photometric survey there is no `is_central`. Find the centres with the CGF and
use them in the BST (`centres="cgf"` does both steps). With real data, pass your own
centres and masses: `compute_hod(cat, -19, centres=df)` where `df` has columns
`galaxy_id` and `log_halo_mass`.

In [ ]:
centres = find_central_galaxies(cat, mlim=-19)
res_cgf = compute_hod(cat, mlim=-19, centres=centres)
plot_hod({"CGF centres": hod_curve(res_cgf), "true centres": hod_curve(res)}, show_true=False)
plt.show()

## 5. Main options

| option | default | meaning |
|---|---|---|
| `factor_r` | 1.0 | aperture radius in units of R200 (scalar or one value per centre) |
| `r_min`, `r_max` | 0.05, 1.25 | aperture clipped to this range [physical Mpc] (`r_max=None`: no clip) |
| `ring` | `"classic"` | `"classic"`: ring at `ring_radius` × aperture; `"fix"`: aperture + 1 … + 2 Mpc |
| `ring_radius` | (1.5, 3.5) | inner/outer ring radius in units of the aperture |
| `mag_cut` | True | count only galaxies brighter than `mlim` at the centre's distance |
| `area` | `"pixel"` | `"pixel"` = original grid estimate; `"analytic"` = exact area ratio |
| `z_max` | original cuts | redshift limit of the centres |
| `save` | None | write the result to `.csv` / `.npz` / `.parquet` |